In [1]:
import pandas as pd

credit = pd.read_csv("data/german_credit_data.csv")

In [2]:
from janitor import clean_names

credit = credit.clean_names()
# credit = clean_names(credit)

In [3]:
credit_with_no_nas = credit.dropna()

## Braki danych

In [4]:
credit["checking_account"].value_counts()

checking_account
little      274
moderate    269
rich         63
Name: count, dtype: int64

In [5]:
checking_account_mode = credit["checking_account"].mode()[0]

credit["checking_account"] = credit["checking_account"].fillna(checking_account_mode)
credit["checking_account"].value_counts()

checking_account
little      668
moderate    269
rich         63
Name: count, dtype: int64

In [6]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="most_frequent")
credit[["saving_accounts"]] = imputer.fit_transform(credit[["saving_accounts"]])

In [7]:
imputer.feature_names_in_    

array(['saving_accounts'], dtype=object)

In [8]:
credit_dedup = credit.drop_duplicates()

## Inżynieria cech

In [10]:
credit["installment_rate"] = credit["credit_amount"] / credit["duration"]
credit["installment_rate"].describe()

count    1000.000000
mean      167.687020
std       153.490959
min        24.055556
25%        89.600000
50%       130.333333
75%       206.183333
max      2482.666667
Name: installment_rate, dtype: float64

In [11]:
credit["credit_amount"].describe()

count     1000.000000
mean      3271.258000
std       2822.736876
min        250.000000
25%       1365.500000
50%       2319.500000
75%       3972.250000
max      18424.000000
Name: credit_amount, dtype: float64

In [12]:
credit["installment_rate_by_checking_account"] = credit.groupby("checking_account")["installment_rate"].transform("mean")

In [14]:
import numpy as np

credit["age_groups"] = pd.cut(credit["age"],
                              bins=[0, 30, 40, 60, np.inf],
                              labels=["=<30", "31-40", "41-60", "60+"])

In [ ]:
credit["purpose"].value_counts()
credit["purpose2"] = credit["purpose"].apply(lambda x: "investment" if x in ["business", "education"] else "consumption")

## One hot encoding

In [19]:
categorical_cols = ["sex", "job", "housing", "saving_accounts", "checking_account", "purpose", "purpose2", "age_groups"]
credit_ohe = pd.get_dummies(credit, columns=categorical_cols, dtype=int)
credit_ohe.head()

,age,credit_amount,duration,risk,installment_rate,installment_rate_by_checking_account,sex_female,sex_male,job_0,job_1,...,purpose_furniture/equipment,purpose_radio/TV,purpose_repairs,purpose_vacation/others,purpose2_consumption,purpose2_investment,age_groups_=<30,age_groups_31-40,age_groups_41-60,age_groups_60+
0,67,1169,6,good,194.833333,166.541882,0,1,0,0,...,0,1,0,0,1,0,0,0,0,1
1,22,5951,48,bad,123.979167,177.341247,1,0,0,0,...,0,1,0,0,1,0,1,0,0,0
2,49,2096,12,good,174.666667,166.541882,0,1,0,1,...,0,0,0,0,0,1,0,0,1,0
3,45,7882,42,good,187.666667,166.541882,0,1,0,0,...,1,0,0,0,1,0,0,0,1,0
4,53,4870,24,bad,202.916667,166.541882,0,1,0,0,...,0,0,0,0,1,0,0,0,1,0


In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe_encoder = OneHotEncoder()
credit_ohe_encoder = ohe_encoder.fit_transform(credit[categorical_cols])
credit_ohe_encoder_df = pd.DataFrame(credit_ohe_encoder.toarray(),
                                     columns=ohe_encoder.get_feature_names_out(categorical_cols))

## Normalizacja cech

In [24]:
numeric_cols = ["age", "credit_amount", "duration", "installment_rate", "installment_rate_by_checking_account"]

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
credit_scaled = scaler.fit_transform(credit[numeric_cols])
credit_scaled_df = pd.DataFrame(credit_scaled, columns=numeric_cols)

## Kodowanie etykiet

In [26]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
risk_le = le.fit_transform(credit["risk"])
risk_le_df = pd.DataFrame(risk_le, columns=["risk"])

## Finalny zbiór danych

In [29]:
credit_final = pd.concat([credit_scaled_df, credit_ohe_encoder_df, risk_le_df], axis=1)

credit_final = credit_final.clean_names()

credit_final.to_csv("data/credit_final.csv", index=False)